<a href="https://colab.research.google.com/github/csu-techhub/QOS-Labs/blob/main/Lab2.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 2 — Complete 5-Node Max-Cut QAOA
**Quantum Optimization and Simulation — QAOA Laboratory Series**

> Qiskit displays measured bitstrings in the order `q_(n-1)...q_0`. When we discuss graph nodes, this notebook often converts them to `q_0...q_(n-1)` using `q0_first(...)`.

## Learning goals
- Build the complete 5-node Max-Cut QAOA circuit used in the lecture.
- Decode measured bitstrings and calculate their cut values.
- Close the hybrid quantum-classical loop.
- Use COBYLA to optimize $\gamma$, $\beta$.
- Plot convergence.
- Compare your gate-level construction with Qiskit's built-in QAOA workflow.

The graph is

\[
E=\{(0,3),(0,4),(1,3),(1,4),(2,3),(2,4)\}.
\]

The maximum cut value is 6; complementary bitstrings represent the same partition.

In [ ]:
# Run this once at the beginning of a fresh Google Colab session.
%pip -q install "pylatexenc" "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-optimization~=0.7" "qiskit-ibm-runtime~=0.46"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_aer.primitives import SamplerV2

SEED = 123
SHOTS = 2048
sampler = SamplerV2(default_shots=SHOTS, seed=SEED)

# Helper functions
def run_counts(qc, shots=SHOTS):
    "Run a measured circuit with Aer SamplerV2 and return counts."
    result = sampler.run([qc], shots=shots).result()
    return result[0].data.meas.get_counts()

def q0_first(qiskit_bitstring):
    "Convert Qiskit's displayed q_(n-1)...q_0 bitstring to q_0...q_(n-1)."
    return qiskit_bitstring.replace(" ", "")[::-1]

In [ ]:
import networkx as nx

edges5 = [(0,3),(0,4),(1,3),(1,4),(2,3),(2,4)]
G = nx.Graph()
G.add_nodes_from(range(5))
G.add_edges_from(edges5)

pos = {0:(0,2), 1:(0,1), 2:(0,0), 3:(1,1.5), 4:(1,0.5)}
nx.draw(G, pos=pos, with_labels=True, node_size=900)
plt.show()

In [ ]:
def build_qaoa_5(gamma, beta, measure=True):
    qc = QuantumCircuit(5)
    qc.h(range(5))

    for i, j in edges5:
        qc.cx(i, j)
        qc.rz(-gamma, j)
        qc.cx(i, j)

    for q in range(5):
        qc.rx(2 * beta, q)

    if measure:
        qc.measure_all()
    return qc

In [ ]:
def cut_value_q0first(bitstring, edges):
    """
    Calculate the cut value for a given bitstring graph partitioning.
    """
    total_cut_edges = 0
    for vertex_a, vertex_b in edges:
        if bitstring[vertex_a] != bitstring[vertex_b]:
            total_cut_edges += 1
    return total_cut_edges

def expected_cut_from_counts(counts, edges):
    """
    Calculate the expected (average) cut value from a distribution of measurement outcomes.
    """
    # Compute total number of measurement samples
    total_shots = sum(counts.values())

    # Handle empty input gracefully to avoid DivisionByZeroError
    if total_shots == 0:
        return 0.0

    weighted_cut_sum = 0.0
    for bitstring_key, count in counts.items():
        formatted_bitstring = q0_first(bitstring_key)
        cut_value = cut_value_q0first(formatted_bitstring, edges)
        weighted_cut_sum += cut_value * count

    expected_value = weighted_cut_sum / total_shots
    return expected_value

## Part A — Run one p=1 circuit

In [ ]:
gamma = 0.68
beta = 0.39

qc = build_qaoa_5(gamma, beta)
display(qc.draw("mpl"))

counts = run_counts(qc, shots=4096)
print("Expected cut =", expected_cut_from_counts(counts, edges5))
plot_histogram(counts)

**Expected qualitative result:** the average cut should be noticeably above the uniform-random value of 3, although a single \(p=1\) layer is not expected to reproduce the deeper optimized lecture result.

**Expected:** the optimal partition appears as the complementary pair `00011` and `11100` when bitstrings are written q0→q4.

## Part B — Run COBYLA

In [ ]:
history = []

def objective_p1(params):
    gamma, beta = params
    qc = build_qaoa_5(gamma, beta)
    counts = run_counts(qc, shots=1024)
    avg_cut = expected_cut_from_counts(counts, edges5)
    history.append(avg_cut)
    return -avg_cut

In [ ]:
from scipy.optimize import minimize

history.clear()
initial_point = np.array([0.3, 0.6])

result = minimize(
    objective_p1,
    x0=initial_point,
    method="COBYLA",
    options={"maxiter": 40, "rhobeg": 0.5}
)

print(result)
print("Best gamma, beta =", result.x)
print("Best sampled average cut ≈", -result.fun)

plt.plot(history, marker=".")
plt.xlabel("Objective evaluation")
plt.ylabel("Average cut")
plt.title("COBYLA convergence")
plt.show()

## Part C — Optional Qiskit's built-in QAOA

In [ ]:
from qiskit_optimization.applications import Maxcut
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_optimization.minimum_eigensolvers import QAOA
from qiskit_optimization.optimizers import COBYLA
from qiskit_aer.primitives import SamplerV2

maxcut = Maxcut(G)
qp = maxcut.to_quadratic_program()
print(qp.prettyprint())

builtin_sampler = SamplerV2(default_shots=2048, seed=SEED)
builtin_qaoa = QAOA(
    sampler=builtin_sampler,
    optimizer=COBYLA(maxiter=60),
    reps=1
)

optimizer = MinimumEigenOptimizer(builtin_qaoa)
result = optimizer.solve(qp)
print(result.prettyprint())
print("Selected partition vector:", result.x)

**Expected:** Qiskit's high-level QAOA should return a feasible high-quality Max-Cut solution. Because QAOA is stochastic and the optimizer is approximate, the exact bitstring and objective value can vary.

## Reflection
1. $U_C$ is the six edge blocks, each implemented as CX–Rz(-$\gamma$)–CX.
2. $U_B$ is the set of five Rx($2\beta$) gates.
3. The high-level call hides construction of the QAOA ansatz, parameter binding, repeated sampler calls, and the classical optimization loop.